# Retraining in the Hanoi Weather Forecasting System

The Hanoi Temperature Forecasting System predicts 5-day future temperatures using machine learning models.  
Once deployed, the model faces several real-world challenges:

- **Model Drift**: The model becomes outdated as data distribution changes.
- **Seasonality Effects**: Weather changes across seasons reduce accuracy.
- **Sensor / Data Changes**: Upstream APIs or data sources may shift.
- **Concept Drift**: Relationships between features and target evolve over time.

To ensure reliable and consistent predictions, an automated **Monitoring + Alerting + Retraining** pipeline is required.

This notebook demonstrates how to build:
- A performance baseline  
- A monitoring system (KPIs + KDIs)  
- Three retraining triggers  
- A fully automated retraining workflow  

## 1. Performance Baseline

A **performance baseline** is a "reference performance level" created during the initial training phase.  
The deployed model is continuously compared against this baseline.

### Why do we need a Baseline?
- Detect performance degradation
- Compare new data vs. old behavior
- Ensure model remains useful in production

### Metrics Used (per forecast horizon)
- **MAE (Mean Absolute Error)**
- **RMSE (Root Mean Squared Error)**
- **R² Score**

We maintain **5 separate baselines**:
- y_temp_1 → forecast for +1 day
- y_temp_2 → forecast for +2 days  
...
- y_temp_5 → forecast for +5 days

### Example Baseline & Threshold
| Model | Baseline MAE | Threshold | Retrain When |
|-------|--------------|-----------|---------------|
| y_temp_1 | 1.5°C | 1.8°C | Rolling MAE > 1.8 |
| y_temp_5 | 2.0°C | 2.5°C | Rolling MAE > 2.5 |

These thresholds are set based on validation performance and acceptable model tolerance.

## 2. Data Collection Pipeline

The system collects two essential data streams every day:

### **2.1 Prediction Logs (from UI / Gradio Application)**

Every time a user requests a forecast, we log:

- `timestamp` → when prediction was made  
- `input_day` → the day user requests prediction  
- `forecast_for` → target date  
- `input features` → humidity, dew point, wind,...  
- `predicted_temp` → model output  
- `model_version` → to support model registry  

**Example logged record:**
**id:** 839123  
**timestamp:** 2025-11-16 10:24  
**input_day:** 2025-11-16  
**forecast_for:** 2025-11-17  
**predicted_temp:** 19.2°C  
**model_version:** v1.3.2  





### **2.2 Ground Truth (Actual Temperature)**

Every night at 23:59 UTC+7, a background script:

1. Calls the **Visual Crossing API**  
2. Downloads yesterday’s actual temperature  
3. Matches records with prediction logs  
4. Saves merged dataset to `pred_vs_actual.csv`

This enables daily performance tracking and KPI computation.


## 3. Monitoring Dashboard

The monitoring system tracks two major categories:

### **3.1 KPI — Model Performance Monitoring**
KPI metrics measure how well the model predicts real-world temperatures:
- 30-day rolling MAE  
- 7-day rolling MAE  
- MAE/RMSE by forecast horizon  
- Actual vs. Predicted comparison plots  

This helps detect early signs of **model performance degradation**.

### **3.2 KDI — Data Quality & Data Drift Monitoring**
KDIs detect changes in input data before the model performance drops:
- Missing value rate  
- Mean and standard deviation  
- Feature distribution shifts  
- Outlier rates  
- Feature correlations changing over time  

We use **Kolmogorov–Smirnov (KS) Test** to detect if the new data distribution differs significantly from training data.

If any KDI deviates over 15% from the original training distribution → **data drift alert**.

## 4. Scheduled Retraining Trigger

Even if:
- No performance degradation  
- No significant drift  

the model still needs **periodic retraining** due to:
- Long-term climate changes  
- Seasonal transitions  
- Feature importance drift  
- Accumulation of new data  

Recommended schedule: **every 6 months**.

## 5. Unified Retraining Trigger Logic

The system triggers retraining if **any** of the following occur:

### **5.1 Performance Degradation**
Rolling MAE > predefined threshold  
→ Model no longer meets expected accuracy

### **5.2 Data Drift**
KS test identifies distribution shift  
→ Input data no longer matches training distribution

### **5.3 Scheduled Retraining**
Time-based refresh  
→ Ensures model adapts to long-term changes

If **one or more triggers** activate → the retraining pipeline starts automatically.

## 6. Retraining Pipeline

> **Note:** This section describes the retraining workflow conceptually, focusing on the steps triggered when retraining is needed. It does **not** include data collection or model-specific details.

### Step 1: Identify Retraining Trigger
- Check if any retraining triggers are active:
  - ⚠️ Performance degradation
  - ⚠️ Data drift
  - ⏰ Scheduled retraining
- Confirm retraining is necessary before initiating the workflow

### Step 2: Execute Retraining Workflow
- Begin the automated retraining sequence
- Ensure workflow is **logged and monitored**
- Maintain reproducibility and consistency throughout

### Step 3: Challenger vs. Champion Evaluation
- Compare the new model (Challenger) with the current production model (Champion)
- Only deploy the Challenger if it demonstrates **better overall performance**

### Step 4: Deploy Updated Model
- Replace the current production model with the approved Challenger
- Update model version and registry
- Resume monitoring for all performance and drift metrics

### Step 5: Continuous Retraining Cycle
- Retraining is a **repeating cycle**:
  1️⃣ Monitor → 2️⃣ Trigger → 3️⃣ Retrain → 4️⃣ Evaluate → 5️⃣ Deploy → repeat
- Ensures the forecasting system remains **accurate, reliable, and adaptive**